# SSM Foundations: Mathematical Basics

> Learn the mathematical foundations of State Space Models

- **Level:** Beginner
- **Time:** ~20 minutes
- **GPU:** Not required for this notebook

In [ ]:
# Imports
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## Continuous-Time SSM

The continuous-time state space model:

$$h'(t) = Ah(t) + Bx(t)$$
$$y(t) = Ch(t)$$

In [ ]:
# Basic SSM with discrete time
class BasicSSM(nn.Module):
    """Basic State Space Model"""
    
    def __init__(self, d_model, d_state):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        
        # A: diagonal, parameterized by log-eigenvalues
        self.A_log = nn.Parameter(torch.randn(d_state))
        
        # Projections
        self.B_proj = nn.Linear(d_model, d_state)
        self.C_proj = nn.Linear(d_state, d_model)
        
        # Delta (step size)
        self.delta = nn.Parameter(torch.ones(d_state) * 0.1)
    
    def forward(self, x):
        batch, seq_len, _ = x.shape
        
        # Discretize A
        A_bar = torch.exp(self.A_log * self.delta)
        
        # Discretization factor
        disc = torch.where(
            torch.abs(self.A_log) > 1e-6,
            (A_bar - 1) / self.A_log,
            self.delta
        )
        
        # Sequential computation
        h = torch.zeros(batch, self.d_state, device=x.device)
        outputs = []
        
        for t in range(seq_len):
            B_t = self.B_proj(x[:, t])
            B_bar = B_t * disc * self.delta
            h = A_bar.unsqueeze(0) * h + B_bar
            outputs.append(self.C_proj(h))
        
        return torch.stack(outputs, dim=1)
# export
BasicSSM = BasicSSM

In [ ]:
# Test basic SSM
ssm = BasicSSM(d_model=32, d_state=64).to(device)
x = torch.randn(2, 16, 32).to(device)
y = ssm(x)
print(f"Input: {x.shape} → Output: {y.shape}")
print(f"Parameters: {sum(p.numel() for p in ssm.parameters()):,}")

## Visualize SSM Behavior

In [ ]:
# Test with different memory types
def create_ssm(memory_type):
    ssm = BasicSSM(d_model=4, d_state=16).to(device)
    with torch.no_grad():
        if memory_type == 'long':
            ssm.A_log.fill_(0.01)
        elif memory_type == 'short':
            ssm.A_log.fill_(-2.0)
    return ssm

# Test input
x_test = torch.zeros(1, 50, 4).to(device)
x_test[0, 10, 0] = 1.0

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for idx, mem_type in enumerate(['short', 'long']):
    ssm = create_ssm(mem_type)
    with torch.no_grad():
        y = ssm(x_test)
    axes[idx].plot(y[0, :, 0].cpu().numpy())
    axes[idx].axvline(x=10, color='k', linestyle=':', alpha=0.5)
    axes[idx].set_title(f'{mem_type.capitalize()} Memory')
    axes[idx].set_xlabel('Position')
    axes[idx].set_ylabel('Output')

plt.tight_layout()
plt.show()

print("✅ Notice: Short memory decays quickly, long memory persists!")

## Summary

Key takeaways:

1. **SSMs** compress sequences into state vectors
2. **A matrix** controls memory (decay vs persist)
3. **Discretization** converts continuous to discrete time
4. **Sequential** computation is O(N) but not parallelizable

---

**Next:** Learn about parallel scan in the next notebook!